In [1]:
import os, io, math, json, random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms
from torch.utils.tensorboard import SummaryWriter

import pyarrow.parquet as pq
from PIL import Image


In [2]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

print("device:", get_device())


device: cuda


In [3]:
OUT_ROOT = os.path.expanduser("~/pcam_outputs_prof_final")
print("OUT_ROOT:", OUT_ROOT)


OUT_ROOT: /home/bontu/pcam_outputs_prof_final


In [4]:
PCAM_ROOT = "/data/pcam/PatchCamelyon/data/"

TRAIN_PARQUETS = [
    os.path.join(PCAM_ROOT, "train-00000-of-00003.parquet"),
    os.path.join(PCAM_ROOT, "train-00001-of-00003.parquet"),
    os.path.join(PCAM_ROOT, "train-00002-of-00003.parquet"),
]
VAL_PARQUET  = os.path.join(PCAM_ROOT, "validation-00000-of-00001.parquet")
TEST_PARQUET = os.path.join(PCAM_ROOT, "test-00000-of-00001.parquet")

print("paths exist:", all(os.path.exists(p) for p in TRAIN_PARQUETS+[VAL_PARQUET, TEST_PARQUET]))


paths exist: True


In [5]:
class PCamShard(Dataset):
    def __init__(self, parquet_path: str, transform):
        self.transform = transform
        self.pf = pq.ParquetFile(parquet_path)
        self.n = self.pf.metadata.num_rows

        self.offsets = []
        off = 0
        for rg in range(self.pf.num_row_groups):
            rows = self.pf.metadata.row_group(rg).num_rows
            self.offsets.append((off, off + rows))
            off += rows

    def __len__(self):
        return self.n

    def _locate(self, idx):
        for rg, (a, b) in enumerate(self.offsets):
            if a <= idx < b:
                return rg, idx - a
        raise IndexError(idx)

    def __getitem__(self, idx):
        rg, j = self._locate(idx)
        t = self.pf.read_row_group(rg, columns=["image", "label"])
        img_cell = t["image"][j].as_py()
        y = int(t["label"][j].as_py())
        img = Image.open(io.BytesIO(img_cell["bytes"])).convert("RGB")
        x = self.transform(img)
        return x, torch.tensor(y, dtype=torch.long)


In [6]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
])
eval_tf = transforms.Compose([transforms.ToTensor()])

train_ds = ConcatDataset([PCamShard(p, train_tf) for p in TRAIN_PARQUETS])
val_ds   = PCamShard(VAL_PARQUET, eval_tf)
test_ds  = PCamShard(TEST_PARQUET, eval_tf)

# workers: start with 8; if your server allows, you can try 12 later.
NUM_WORKERS = 8

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)

print("sizes:", len(train_ds), len(val_ds), len(test_ds))


sizes: 262144 32768 32768


In [7]:
device = get_device()
vx, vy = next(iter(val_loader))
vx, vy = vx.to(device, non_blocking=True), vy.to(device, non_blocking=True)
print("fixed val batch:", vx.shape, vy.shape)


fixed val batch: torch.Size([64, 3, 96, 96]) torch.Size([64])


In [8]:
def metrics_from_logits_binary(logits: np.ndarray, y_true: np.ndarray):
    y_true = y_true.astype(int)

    y_pred = logits.argmax(axis=1)
    acc = float((y_pred == y_true).mean())

    y_hat = (logits[:, 1] >= logits[:, 0]).astype(int)
    tp = int(((y_hat == 1) & (y_true == 1)).sum())
    fp = int(((y_hat == 1) & (y_true == 0)).sum())
    fn = int(((y_hat == 0) & (y_true == 1)).sum())

    if tp == 0:
        f1 = 0.0
    else:
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = float(2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    # ROC-AUC (rank-based)
    z = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
    scores = probs[:, 1]

    pos = scores[y_true == 1]
    neg = scores[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        roc_auc = float("nan")
    else:
        all_scores = np.concatenate([pos, neg])
        ranks = all_scores.argsort().argsort() + 1
        sum_ranks_pos = ranks[:len(pos)].sum()
        n_pos, n_neg = len(pos), len(neg)
        roc_auc = float((sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

    return acc, f1, roc_auc


In [9]:
from torch.amp import autocast, GradScaler
scaler = GradScaler("cuda")

def eval_loss_acc(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    losses = []
    correct = 0
    total = 0

    with torch.inference_mode(), autocast(device_type="cuda"):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            losses.append(float(loss.item()))
            pred = logits.argmax(dim=1)
            correct += int((pred == y).sum().item())
            total += int(y.numel())

    return float(np.mean(losses)), correct / total


In [10]:
def eval_all_metrics(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    logits_all = []
    y_all = []
    losses = []

    with torch.inference_mode(), autocast(device_type="cuda"):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            losses.append(float(loss.item()))
            logits_all.append(logits.detach().cpu().numpy())
            y_all.append(y.detach().cpu().numpy())

    logits_np = np.concatenate(logits_all, axis=0)
    y_np = np.concatenate(y_all, axis=0)
    acc, f1, roc_auc = metrics_from_logits_binary(logits_np, y_np)
    return float(np.mean(losses)), acc, f1, roc_auc


In [11]:
def make_scheduler(name, optimizer, total_epochs=60, base_lr=1e-3, min_lr=1e-6):
    name = name.lower()
    if name == "constant":
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: 1.0)
    if name == "step":
        return torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 40], gamma=0.1)
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=min_lr)
    if name == "warm_restarts":
        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=1, eta_min=min_lr)
    if name == "warmup_cosine":
        warmup_epochs = 5
        warmup_start = 1e-5
        def lr_lambda(epoch):
            e = epoch + 1
            if e <= warmup_epochs:
                lr = warmup_start + (base_lr - warmup_start) * (e - 1) / (warmup_epochs - 1)
                return lr / base_lr
            t = (e - warmup_epochs) / (total_epochs - warmup_epochs)
            lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * t))
            return lr / base_lr
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    raise ValueError(name)


In [12]:
def grad_norms_on_fixed_batch(model, criterion, x, y):
    model.train(True)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    norms = []
    for p in model.parameters():
        if p.grad is not None:
            norms.append(float(p.grad.detach().data.norm(2).item()))
    return norms


In [13]:
def save_checkpoint(path, epoch, model, optimizer, scheduler):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
    }, path)

def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location="cuda")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    return int(ckpt["epoch"])


In [14]:
def run_one(schedule="constant", seed=42, epochs=60, out_root=OUT_ROOT):
    set_seed(seed)
    device = get_device()

    run_dir = os.path.join(out_root, "runs", schedule, f"seed{seed}")
    os.makedirs(run_dir, exist_ok=True)

    tb_dir = os.path.join(run_dir, "tensorboard")
    os.makedirs(tb_dir, exist_ok=True)

    ckpt_path  = os.path.join(run_dir, "checkpoint.pt")
    hist_path  = os.path.join(run_dir, "history.json")
    grad_path  = os.path.join(run_dir, "gradnorms.json")
    final_path = os.path.join(run_dir, "final_metrics.json")

    writer = SummaryWriter(log_dir=tb_dir)

    model = models.resnet34(weights=None)  # from scratch
    model.fc = nn.Linear(model.fc.in_features, 2)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
    scheduler = make_scheduler(schedule, optimizer, total_epochs=epochs)

    history = []
    gradnorms = {}

    start_epoch = 1
    if os.path.exists(hist_path):
        try:
            history = json.load(open(hist_path))
        except:
            history = []

    if os.path.exists(ckpt_path):
        start_epoch = load_checkpoint(ckpt_path, model, optimizer, scheduler) + 1
        print(f"Resuming {schedule} seed{seed} from epoch {start_epoch}")
    else:
        print(f"Starting {schedule} seed{seed} from epoch 1")

    for epoch in range(start_epoch, epochs + 1):
        # ---- TRAIN ----
        model.train()
        train_losses = []
        correct = 0
        total = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda"):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.item()))
            pred = logits.argmax(dim=1)
            correct += int((pred == y).sum().item())
            total += int(y.numel())

        tr_loss = float(np.mean(train_losses))
        tr_acc = correct / total

        scheduler.step()
        lr = float(optimizer.param_groups[0]["lr"])

        # ---- VALIDATION (FAST: loss+acc each epoch) ----
        va_loss, va_acc = eval_loss_acc(model, val_loader, device)

        # ---- GRAD NORMS (spec: constant & cosine at 1/20/40/60) ----
        if schedule in ("constant", "cosine") and epoch in (1, 20, 40, 60):
            gradnorms[str(epoch)] = grad_norms_on_fixed_batch(model, criterion, vx, vy)

        # ---- LOGGING ----
        writer.add_scalar("loss/train", tr_loss, epoch)
        writer.add_scalar("acc/train", tr_acc, epoch)
        writer.add_scalar("loss/val", va_loss, epoch)
        writer.add_scalar("acc/val", va_acc, epoch)
        writer.add_scalar("lr", lr, epoch)

        history.append({
            "epoch": epoch,
            "lr": lr,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": va_loss,
            "val_acc": va_acc
        })

        if epoch % 5 == 0:
            save_checkpoint(ckpt_path, epoch, model, optimizer, scheduler)
            json.dump(history, open(hist_path, "w"), indent=2)
            json.dump(gradnorms, open(grad_path, "w"), indent=2)
            print(f"[{schedule} seed{seed}] epoch {epoch:02d} lr={lr:.2e} val_acc={va_acc:.4f} (saved)")

        if epoch in (1, 2, 3, 5, 10, 20, 40, epochs):
            print(f"[{schedule} seed{seed}] epoch {epoch:02d} lr={lr:.2e} val_acc={va_acc:.4f}")

    # ---- FINAL METRICS (SPEC: acc, f1, roc_auc for val+test) ----
    final_val_loss, final_val_acc, final_val_f1, final_val_auc = eval_all_metrics(model, val_loader, device)
    te_loss, te_acc, te_f1, te_auc = eval_all_metrics(model, test_loader, device)

    final_metrics = {
        "schedule": schedule,
        "seed": seed,
        "final_epoch": epochs,
        "final_val": {"loss": final_val_loss, "acc": final_val_acc, "f1": final_val_f1, "roc_auc": final_val_auc},
        "final_test": {"loss": te_loss, "acc": te_acc, "f1": te_f1, "roc_auc": te_auc}
    }
    json.dump(final_metrics, open(final_path, "w"), indent=2)
    json.dump(history, open(hist_path, "w"), indent=2)
    json.dump(gradnorms, open(grad_path, "w"), indent=2)

    writer.close()
    return run_dir


In [15]:
# sanity (fast)
_ = run_one(schedule="constant", seed=42, epochs=2, out_root=OUT_ROOT)

# real (uncomment after sanity)
# _ = run_one(schedule="constant", seed=42, epochs=60, out_root=OUT_ROOT)


Starting constant seed42 from epoch 1
[constant seed42] epoch 01 lr=1.00e-03 val_acc=0.7926
[constant seed42] epoch 02 lr=1.00e-03 val_acc=0.8235


In [ ]:
# sanity (fast)
#_ = run_one(schedule="constant", seed=42, epochs=2, out_root=OUT_ROOT)

# real (uncomment after sanity)
_ = run_one(schedule="constant", seed=42, epochs=60, out_root=OUT_ROOT)

Starting constant seed42 from epoch 1


In [2]:
import os
OUT_ROOT = os.path.expanduser("~/pcam_outputs_prof_final")
print("OUT_ROOT =", OUT_ROOT)


OUT_ROOT = /home/bontu/pcam_outputs_prof_final


In [3]:
run_dir = os.path.join(OUT_ROOT, "runs", "constant", "seed42")
print("Run dir:", run_dir)

for f in ["history.json", "final_metrics.json", "gradnorms.json", "checkpoint.pt"]:
    print(f, "exists?", os.path.exists(os.path.join(run_dir, f)))


Run dir: /home/bontu/pcam_outputs_prof_final/runs/constant/seed42
history.json exists? True
final_metrics.json exists? True
gradnorms.json exists? True
checkpoint.pt exists? True


In [19]:
print(run_one)


<function run_one at 0x7d40fc5ca020>


In [20]:
_ = run_one(schedule="constant", seed=43, epochs=60, out_root=OUT_ROOT)


Starting constant seed43 from epoch 1
[constant seed43] epoch 01 lr=1.00e-03 val_acc=0.8105
[constant seed43] epoch 02 lr=1.00e-03 val_acc=0.7872
[constant seed43] epoch 03 lr=1.00e-03 val_acc=0.7124
[constant seed43] epoch 05 lr=1.00e-03 val_acc=0.7560 (saved)
[constant seed43] epoch 05 lr=1.00e-03 val_acc=0.7560
[constant seed43] epoch 10 lr=1.00e-03 val_acc=0.6994 (saved)
[constant seed43] epoch 10 lr=1.00e-03 val_acc=0.6994
[constant seed43] epoch 15 lr=1.00e-03 val_acc=0.8073 (saved)
[constant seed43] epoch 20 lr=1.00e-03 val_acc=0.7524 (saved)
[constant seed43] epoch 20 lr=1.00e-03 val_acc=0.7524
[constant seed43] epoch 25 lr=1.00e-03 val_acc=0.7888 (saved)
[constant seed43] epoch 30 lr=1.00e-03 val_acc=0.8323 (saved)
[constant seed43] epoch 35 lr=1.00e-03 val_acc=0.8122 (saved)
[constant seed43] epoch 40 lr=1.00e-03 val_acc=0.8367 (saved)
[constant seed43] epoch 40 lr=1.00e-03 val_acc=0.8367
[constant seed43] epoch 45 lr=1.00e-03 val_acc=0.8158 (saved)
[constant seed43] epoch 50

In [21]:
_ = run_one(schedule="constant", seed=44, epochs=60, out_root=OUT_ROOT)


Starting constant seed44 from epoch 1
[constant seed44] epoch 01 lr=1.00e-03 val_acc=0.7652
[constant seed44] epoch 02 lr=1.00e-03 val_acc=0.8373
[constant seed44] epoch 03 lr=1.00e-03 val_acc=0.8095
[constant seed44] epoch 05 lr=1.00e-03 val_acc=0.8567 (saved)
[constant seed44] epoch 05 lr=1.00e-03 val_acc=0.8567
[constant seed44] epoch 10 lr=1.00e-03 val_acc=0.7756 (saved)
[constant seed44] epoch 10 lr=1.00e-03 val_acc=0.7756
[constant seed44] epoch 15 lr=1.00e-03 val_acc=0.8806 (saved)
[constant seed44] epoch 20 lr=1.00e-03 val_acc=0.8565 (saved)
[constant seed44] epoch 20 lr=1.00e-03 val_acc=0.8565
[constant seed44] epoch 25 lr=1.00e-03 val_acc=0.8404 (saved)
[constant seed44] epoch 30 lr=1.00e-03 val_acc=0.7939 (saved)
[constant seed44] epoch 35 lr=1.00e-03 val_acc=0.8613 (saved)
[constant seed44] epoch 40 lr=1.00e-03 val_acc=0.8575 (saved)
[constant seed44] epoch 40 lr=1.00e-03 val_acc=0.8575
[constant seed44] epoch 45 lr=1.00e-03 val_acc=0.8482 (saved)
[constant seed44] epoch 50

In [15]:
_ = run_one(schedule="step", seed=42, epochs=60, out_root=OUT_ROOT)


Starting step seed42 from epoch 1
[step seed42] epoch 01 lr=1.00e-03 val_acc=0.7926
[step seed42] epoch 02 lr=1.00e-03 val_acc=0.8235
[step seed42] epoch 03 lr=1.00e-03 val_acc=0.8144
[step seed42] epoch 05 lr=1.00e-03 val_acc=0.8263 (saved)
[step seed42] epoch 05 lr=1.00e-03 val_acc=0.8263
[step seed42] epoch 10 lr=1.00e-03 val_acc=0.8715 (saved)
[step seed42] epoch 10 lr=1.00e-03 val_acc=0.8715
[step seed42] epoch 15 lr=1.00e-03 val_acc=0.8631 (saved)
[step seed42] epoch 20 lr=1.00e-04 val_acc=0.8716 (saved)
[step seed42] epoch 20 lr=1.00e-04 val_acc=0.8716
[step seed42] epoch 25 lr=1.00e-04 val_acc=0.8520 (saved)
[step seed42] epoch 30 lr=1.00e-04 val_acc=0.8376 (saved)
[step seed42] epoch 35 lr=1.00e-04 val_acc=0.8142 (saved)
[step seed42] epoch 40 lr=1.00e-05 val_acc=0.8469 (saved)
[step seed42] epoch 40 lr=1.00e-05 val_acc=0.8469
[step seed42] epoch 45 lr=1.00e-05 val_acc=0.8395 (saved)
[step seed42] epoch 50 lr=1.00e-05 val_acc=0.8415 (saved)
[step seed42] epoch 55 lr=1.00e-05 v

In [16]:
_ = run_one(schedule="step", seed=43, epochs=60, out_root=OUT_ROOT)


Starting step seed43 from epoch 1
[step seed43] epoch 01 lr=1.00e-03 val_acc=0.6413
[step seed43] epoch 02 lr=1.00e-03 val_acc=0.7493
[step seed43] epoch 03 lr=1.00e-03 val_acc=0.7931
[step seed43] epoch 05 lr=1.00e-03 val_acc=0.8720 (saved)
[step seed43] epoch 05 lr=1.00e-03 val_acc=0.8720
[step seed43] epoch 10 lr=1.00e-03 val_acc=0.8721 (saved)
[step seed43] epoch 10 lr=1.00e-03 val_acc=0.8721
[step seed43] epoch 15 lr=1.00e-03 val_acc=0.7707 (saved)
[step seed43] epoch 20 lr=1.00e-04 val_acc=0.8478 (saved)
[step seed43] epoch 20 lr=1.00e-04 val_acc=0.8478
[step seed43] epoch 25 lr=1.00e-04 val_acc=0.8534 (saved)
[step seed43] epoch 30 lr=1.00e-04 val_acc=0.8367 (saved)
[step seed43] epoch 35 lr=1.00e-04 val_acc=0.8333 (saved)
[step seed43] epoch 40 lr=1.00e-05 val_acc=0.8423 (saved)
[step seed43] epoch 40 lr=1.00e-05 val_acc=0.8423
[step seed43] epoch 45 lr=1.00e-05 val_acc=0.8449 (saved)
[step seed43] epoch 50 lr=1.00e-05 val_acc=0.8462 (saved)
[step seed43] epoch 55 lr=1.00e-05 v

In [17]:
_ = run_one(schedule="step", seed=44, epochs=60, out_root=OUT_ROOT)


Starting step seed44 from epoch 1
[step seed44] epoch 01 lr=1.00e-03 val_acc=0.8043
[step seed44] epoch 02 lr=1.00e-03 val_acc=0.8192
[step seed44] epoch 03 lr=1.00e-03 val_acc=0.7680
[step seed44] epoch 05 lr=1.00e-03 val_acc=0.8335 (saved)
[step seed44] epoch 05 lr=1.00e-03 val_acc=0.8335
[step seed44] epoch 10 lr=1.00e-03 val_acc=0.7458 (saved)
[step seed44] epoch 10 lr=1.00e-03 val_acc=0.7458
[step seed44] epoch 15 lr=1.00e-03 val_acc=0.8328 (saved)
[step seed44] epoch 20 lr=1.00e-04 val_acc=0.7830 (saved)
[step seed44] epoch 20 lr=1.00e-04 val_acc=0.7830
[step seed44] epoch 25 lr=1.00e-04 val_acc=0.8422 (saved)
[step seed44] epoch 30 lr=1.00e-04 val_acc=0.8399 (saved)
[step seed44] epoch 35 lr=1.00e-04 val_acc=0.8310 (saved)
[step seed44] epoch 40 lr=1.00e-05 val_acc=0.8261 (saved)
[step seed44] epoch 40 lr=1.00e-05 val_acc=0.8261
[step seed44] epoch 45 lr=1.00e-05 val_acc=0.8318 (saved)
[step seed44] epoch 50 lr=1.00e-05 val_acc=0.8352 (saved)
[step seed44] epoch 55 lr=1.00e-05 v

In [18]:
_ = run_one(schedule="cosine", seed=42, epochs=60, out_root=OUT_ROOT)



Starting cosine seed42 from epoch 1
[cosine seed42] epoch 01 lr=9.99e-04 val_acc=0.7553
[cosine seed42] epoch 02 lr=9.97e-04 val_acc=0.7328
[cosine seed42] epoch 03 lr=9.94e-04 val_acc=0.6844
[cosine seed42] epoch 05 lr=9.83e-04 val_acc=0.8540 (saved)
[cosine seed42] epoch 05 lr=9.83e-04 val_acc=0.8540
[cosine seed42] epoch 10 lr=9.33e-04 val_acc=0.7248 (saved)
[cosine seed42] epoch 10 lr=9.33e-04 val_acc=0.7248
[cosine seed42] epoch 15 lr=8.54e-04 val_acc=0.8470 (saved)
[cosine seed42] epoch 20 lr=7.50e-04 val_acc=0.8080 (saved)
[cosine seed42] epoch 20 lr=7.50e-04 val_acc=0.8080
[cosine seed42] epoch 25 lr=6.30e-04 val_acc=0.8532 (saved)
[cosine seed42] epoch 30 lr=5.01e-04 val_acc=0.8253 (saved)
[cosine seed42] epoch 35 lr=3.71e-04 val_acc=0.8372 (saved)
[cosine seed42] epoch 40 lr=2.51e-04 val_acc=0.8404 (saved)
[cosine seed42] epoch 40 lr=2.51e-04 val_acc=0.8404
[cosine seed42] epoch 45 lr=1.47e-04 val_acc=0.8396 (saved)
[cosine seed42] epoch 50 lr=6.79e-05 val_acc=0.8315 (saved)


In [19]:
_ = run_one(schedule="cosine", seed=43, epochs=60, out_root=OUT_ROOT)

Starting cosine seed43 from epoch 1
[cosine seed43] epoch 01 lr=9.99e-04 val_acc=0.7169
[cosine seed43] epoch 02 lr=9.97e-04 val_acc=0.8080
[cosine seed43] epoch 03 lr=9.94e-04 val_acc=0.8483
[cosine seed43] epoch 05 lr=9.83e-04 val_acc=0.7961 (saved)
[cosine seed43] epoch 05 lr=9.83e-04 val_acc=0.7961
[cosine seed43] epoch 10 lr=9.33e-04 val_acc=0.8549 (saved)
[cosine seed43] epoch 10 lr=9.33e-04 val_acc=0.8549
[cosine seed43] epoch 15 lr=8.54e-04 val_acc=0.8230 (saved)
[cosine seed43] epoch 20 lr=7.50e-04 val_acc=0.8610 (saved)
[cosine seed43] epoch 20 lr=7.50e-04 val_acc=0.8610
[cosine seed43] epoch 25 lr=6.30e-04 val_acc=0.8071 (saved)
[cosine seed43] epoch 30 lr=5.01e-04 val_acc=0.8319 (saved)
[cosine seed43] epoch 35 lr=3.71e-04 val_acc=0.8139 (saved)
[cosine seed43] epoch 40 lr=2.51e-04 val_acc=0.8393 (saved)
[cosine seed43] epoch 40 lr=2.51e-04 val_acc=0.8393
[cosine seed43] epoch 45 lr=1.47e-04 val_acc=0.8500 (saved)
[cosine seed43] epoch 50 lr=6.79e-05 val_acc=0.8369 (saved)


In [20]:
_ = run_one(schedule="cosine", seed=44, epochs=60, out_root=OUT_ROOT)

Starting cosine seed44 from epoch 1
[cosine seed44] epoch 01 lr=9.99e-04 val_acc=0.7595
[cosine seed44] epoch 02 lr=9.97e-04 val_acc=0.6974
[cosine seed44] epoch 03 lr=9.94e-04 val_acc=0.7140
[cosine seed44] epoch 05 lr=9.83e-04 val_acc=0.8280 (saved)
[cosine seed44] epoch 05 lr=9.83e-04 val_acc=0.8280
[cosine seed44] epoch 10 lr=9.33e-04 val_acc=0.8717 (saved)
[cosine seed44] epoch 10 lr=9.33e-04 val_acc=0.8717
[cosine seed44] epoch 15 lr=8.54e-04 val_acc=0.8824 (saved)
[cosine seed44] epoch 20 lr=7.50e-04 val_acc=0.8537 (saved)
[cosine seed44] epoch 20 lr=7.50e-04 val_acc=0.8537
[cosine seed44] epoch 25 lr=6.30e-04 val_acc=0.8267 (saved)
[cosine seed44] epoch 30 lr=5.01e-04 val_acc=0.8341 (saved)
[cosine seed44] epoch 35 lr=3.71e-04 val_acc=0.8645 (saved)
[cosine seed44] epoch 40 lr=2.51e-04 val_acc=0.8491 (saved)
[cosine seed44] epoch 40 lr=2.51e-04 val_acc=0.8491
[cosine seed44] epoch 45 lr=1.47e-04 val_acc=0.8442 (saved)
[cosine seed44] epoch 50 lr=6.79e-05 val_acc=0.8454 (saved)


In [21]:
_ = run_one(schedule="warm_restarts", seed=42, epochs=60, out_root=OUT_ROOT)
_ = run_one(schedule="warm_restarts", seed=43, epochs=60, out_root=OUT_ROOT)
_ = run_one(schedule="warm_restarts", seed=44, epochs=60, out_root=OUT_ROOT)




Starting warm_restarts seed42 from epoch 1
[warm_restarts seed42] epoch 01 lr=9.89e-04 val_acc=0.7636
[warm_restarts seed42] epoch 02 lr=9.57e-04 val_acc=0.8111
[warm_restarts seed42] epoch 03 lr=9.05e-04 val_acc=0.8611
[warm_restarts seed42] epoch 05 lr=7.50e-04 val_acc=0.8367 (saved)
[warm_restarts seed42] epoch 05 lr=7.50e-04 val_acc=0.8367
[warm_restarts seed42] epoch 10 lr=2.51e-04 val_acc=0.7599 (saved)
[warm_restarts seed42] epoch 10 lr=2.51e-04 val_acc=0.7599
[warm_restarts seed42] epoch 15 lr=1.00e-03 val_acc=0.8568 (saved)
[warm_restarts seed42] epoch 20 lr=7.50e-04 val_acc=0.8331 (saved)
[warm_restarts seed42] epoch 20 lr=7.50e-04 val_acc=0.8331
[warm_restarts seed42] epoch 25 lr=2.51e-04 val_acc=0.8354 (saved)
[warm_restarts seed42] epoch 30 lr=1.00e-03 val_acc=0.8428 (saved)
[warm_restarts seed42] epoch 35 lr=7.50e-04 val_acc=0.8312 (saved)
[warm_restarts seed42] epoch 40 lr=2.51e-04 val_acc=0.8356 (saved)
[warm_restarts seed42] epoch 40 lr=2.51e-04 val_acc=0.8356
[warm_re

In [22]:
_ = run_one(schedule="warmup_cosine", seed=42, epochs=60, out_root=OUT_ROOT)
_ = run_one(schedule="warmup_cosine", seed=43, epochs=60, out_root=OUT_ROOT)
_ = run_one(schedule="warmup_cosine", seed=44, epochs=60, out_root=OUT_ROOT)


Starting warmup_cosine seed42 from epoch 1
[warmup_cosine seed42] epoch 01 lr=2.58e-04 val_acc=0.7596
[warmup_cosine seed42] epoch 02 lr=5.05e-04 val_acc=0.8166
[warmup_cosine seed42] epoch 03 lr=7.53e-04 val_acc=0.8188
[warmup_cosine seed42] epoch 05 lr=9.99e-04 val_acc=0.8465 (saved)
[warmup_cosine seed42] epoch 05 lr=9.99e-04 val_acc=0.8465
[warmup_cosine seed42] epoch 10 lr=9.71e-04 val_acc=0.7065 (saved)
[warmup_cosine seed42] epoch 10 lr=9.71e-04 val_acc=0.7065
[warmup_cosine seed42] epoch 15 lr=9.05e-04 val_acc=0.8891 (saved)
[warmup_cosine seed42] epoch 20 lr=8.06e-04 val_acc=0.8036 (saved)
[warmup_cosine seed42] epoch 20 lr=8.06e-04 val_acc=0.8036
[warmup_cosine seed42] epoch 25 lr=6.82e-04 val_acc=0.8603 (saved)
[warmup_cosine seed42] epoch 30 lr=5.43e-04 val_acc=0.8111 (saved)
[warmup_cosine seed42] epoch 35 lr=4.01e-04 val_acc=0.8158 (saved)
[warmup_cosine seed42] epoch 40 lr=2.67e-04 val_acc=0.8447 (saved)
[warmup_cosine seed42] epoch 40 lr=2.67e-04 val_acc=0.8447
[warmup_